# SE-801 Lecture 7: Optimization Algorithms and Learning Dynamics

This notebook measures gradient noise, visualizes optimizer paths on an
ill-conditioned quadratic, and compares three optimizers under one frozen
classification protocol. All data are generated locally.

In [1]:
from __future__ import annotations

import copy
import json
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

SEED = 20261019
ROOT = Path.cwd()
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
print("PyTorch:", torch.__version__)
print("Seed:", SEED)

PyTorch: 2.14.0+cu130
Seed: 20261019


## 1. Optimizer trajectories on a poorly conditioned quadratic

The objective is $f(x,y)=\tfrac12(0.2x^2+8y^2)$. Its curvature ratio is 40.
We implement the update rules directly so every state variable is visible.

In [2]:
def objective(theta: torch.Tensor) -> torch.Tensor:
    return 0.5 * (0.2 * theta[0] ** 2 + 8.0 * theta[1] ** 2)


def gradient(theta: torch.Tensor) -> torch.Tensor:
    return torch.tensor([0.2 * theta[0], 8.0 * theta[1]], dtype=torch.float64)


def quadratic_path(name: str, steps: int = 35) -> list[list[float]]:
    theta = torch.tensor([4.0, 1.0], dtype=torch.float64)
    velocity = torch.zeros_like(theta)
    first = torch.zeros_like(theta)
    second = torch.zeros_like(theta)
    path = [theta.tolist()]
    for step in range(1, steps + 1):
        g = gradient(theta)
        if name == "SGD":
            theta = theta - 0.12 * g
        elif name == "Momentum":
            velocity = 0.9 * velocity + g
            theta = theta - 0.05 * velocity
        elif name == "Adam":
            first = 0.9 * first + 0.1 * g
            second = 0.999 * second + 0.001 * g.square()
            m_hat = first / (1 - 0.9**step)
            s_hat = second / (1 - 0.999**step)
            theta = theta - 0.18 * m_hat / (s_hat.sqrt() + 1e-8)
        else:
            raise ValueError(name)
        path.append(theta.tolist())
    return path


quadratic_paths = {name: quadratic_path(name) for name in ["SGD", "Momentum", "Adam"]}
quadratic_final_loss = {
    name: float(objective(torch.tensor(path[-1], dtype=torch.float64)))
    for name, path in quadratic_paths.items()
}
print("Quadratic final losses:", quadratic_final_loss)

grid_x = np.linspace(-4.5, 4.5, 240)
grid_y = np.linspace(-1.4, 1.4, 240)
xx, yy = np.meshgrid(grid_x, grid_y)
zz = 0.5 * (0.2 * xx**2 + 8 * yy**2)
fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
ax.contour(xx, yy, zz, levels=np.geomspace(0.02, 8, 12), colors="#B9C8D1", linewidths=0.8)
for name, path in quadratic_paths.items():
    values = np.asarray(path)
    ax.plot(values[:, 0], values[:, 1], marker="o", markersize=2.5, linewidth=1.8, label=name)
ax.set(xlabel="x", ylabel="y", title="Optimizer paths on an ill-conditioned quadratic")
ax.legend()
fig.savefig(OUTPUT_DIR / "quadratic_optimizer_paths.png", dpi=170)
plt.close(fig)

Quadratic final losses: {'SGD': 0.29215089380341386, 'Momentum': 0.04367601803914182, 'Adam': 0.024376165857133147}


## 2. Synthetic three-class spiral data

Training, validation, and test sets are generated independently. The test set
remains unused until every optimizer recipe has selected its validation
checkpoint.

In [3]:
def make_spiral(points_per_class: int, *, seed: int, noise: float = 0.18):
    generator = torch.Generator().manual_seed(seed)
    xs, ys = [], []
    for class_id in range(3):
        radius = torch.linspace(0.12, 1.0, points_per_class)
        angle = torch.linspace(0.0, 1.45 * math.pi, points_per_class) + class_id * 2 * math.pi / 3
        angle = angle + noise * torch.randn(points_per_class, generator=generator)
        points = torch.stack([radius * torch.cos(angle), radius * torch.sin(angle)], dim=1)
        xs.append(points)
        ys.append(torch.full((points_per_class,), class_id, dtype=torch.long))
    x, y = torch.cat(xs), torch.cat(ys)
    order = torch.randperm(len(y), generator=generator)
    return x[order], y[order]


train_x, train_y = make_spiral(180, seed=SEED + 1)
val_x, val_y = make_spiral(60, seed=SEED + 2)
test_x, test_y = make_spiral(60, seed=SEED + 3)
print("Split shapes:", train_x.shape, val_x.shape, test_x.shape)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.6), constrained_layout=True)
for axis, name, x, y in zip(axes, ["train", "validation", "test"], [train_x, val_x, test_x], [train_y, val_y, test_y]):
    axis.scatter(x[:, 0], x[:, 1], c=y, cmap="viridis", s=11, alpha=0.8)
    axis.set(title=name, xticks=[], yticks=[], aspect="equal")
fig.savefig(OUTPUT_DIR / "spiral_splits.png", dpi=170)
plt.close(fig)

Split shapes: torch.Size([540, 2]) torch.Size([180, 2]) torch.Size([180, 2])


## 3. Gradient variability at fixed parameters

We compare sampled minibatch gradients with the full training gradient without
changing the model. The reported ratio is the mean L2 deviation divided by the
full-gradient norm.

In [4]:
class SpiralMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, 32), nn.Tanh(), nn.Linear(32, 32), nn.Tanh(), nn.Linear(32, 3))

    def forward(self, x):
        return self.net(x)


torch.manual_seed(SEED + 10)
noise_model = SpiralMLP()


def flat_gradient(model: nn.Module, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    model.zero_grad(set_to_none=True)
    F.cross_entropy(model(x), y).backward()
    return torch.cat([p.grad.detach().flatten() for p in model.parameters()])


full_gradient = flat_gradient(noise_model, train_x, train_y)
full_norm = float(full_gradient.norm())
batch_noise = {}
for batch_size in [1, 16, 64, len(train_y)]:
    deviations = []
    cosines = []
    generator = torch.Generator().manual_seed(SEED + batch_size)
    repeats = 1 if batch_size == len(train_y) else 48
    for _ in range(repeats):
        if batch_size == len(train_y):
            index = torch.arange(len(train_y))
        else:
            index = torch.randperm(len(train_y), generator=generator)[:batch_size]
        estimate = flat_gradient(noise_model, train_x[index], train_y[index])
        deviations.append(float((estimate - full_gradient).norm() / full_gradient.norm()))
        cosines.append(float(F.cosine_similarity(estimate, full_gradient, dim=0)))
    batch_noise[str(batch_size)] = {
        "relative_deviation_mean": float(np.mean(deviations)),
        "cosine_mean": float(np.mean(cosines)),
    }
print(json.dumps(batch_noise, indent=2))

{
  "1": {
    "relative_deviation_mean": 9.254812637964884,
    "cosine_mean": 0.10105084699656193
  },
  "16": {
    "relative_deviation_mean": 1.9768557337423165,
    "cosine_mean": 0.508456829469651
  },
  "64": {
    "relative_deviation_mean": 0.8641290999948978,
    "cosine_mean": 0.7608664315193892
  },
  "540": {
    "relative_deviation_mean": 0.0,
    "cosine_mean": 0.9999997615814209
  }
}


## 4. Controlled optimizer comparison

Every run starts from the same parameter tensors and sees the same epoch-wise
batch order. Learning rates are registered choices, and all methods receive an
80-epoch budget.

In [5]:
torch.manual_seed(SEED + 20)
initial_model = SpiralMLP()
initial_state = copy.deepcopy(initial_model.state_dict())


def accuracy(logits, targets):
    return float((logits.argmax(dim=1) == targets).float().mean())


def make_optimizer(name, model):
    if name == "SGD":
        return torch.optim.SGD(model.parameters(), lr=0.08)
    if name == "Momentum":
        return torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
    if name == "Adam":
        return torch.optim.Adam(model.parameters(), lr=0.01)
    raise ValueError(name)


def train_recipe(name: str):
    model = SpiralMLP()
    model.load_state_dict(initial_state)
    optimizer = make_optimizer(name, model)
    generator = torch.Generator().manual_seed(SEED + 30)
    loader = DataLoader(TensorDataset(train_x, train_y), batch_size=54, shuffle=True, generator=generator)
    best = None
    best_loss = float("inf")
    history = []
    for epoch in range(1, 81):
        model.train()
        for x, y in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(x), y)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            train_logits = model(train_x)
            val_logits = model(val_x)
            row = {
                "epoch": epoch,
                "train_loss": float(F.cross_entropy(train_logits, train_y)),
                "validation_loss": float(F.cross_entropy(val_logits, val_y)),
                "train_accuracy": accuracy(train_logits, train_y),
                "validation_accuracy": accuracy(val_logits, val_y),
            }
        history.append(row)
        if row["validation_loss"] < best_loss:
            best_loss = row["validation_loss"]
            best = {"epoch": epoch, "state": copy.deepcopy(model.state_dict())}
    model.load_state_dict(best["state"])
    model.eval()
    with torch.no_grad():
        test_logits = model(test_x)
    return {
        "history": history,
        "best_epoch": best["epoch"],
        "best_validation_loss": best_loss,
        "test_loss": float(F.cross_entropy(test_logits, test_y)),
        "test_accuracy": accuracy(test_logits, test_y),
    }


optimizer_results = {name: train_recipe(name) for name in ["SGD", "Momentum", "Adam"]}
for name, result in optimizer_results.items():
    print(name, {key: value for key, value in result.items() if key != "history"})

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
colors = {"SGD": "#A8483D", "Momentum": "#B87521", "Adam": "#246B8E"}
for name, result in optimizer_results.items():
    epochs = [row["epoch"] for row in result["history"]]
    axes[0].plot(epochs, [row["validation_loss"] for row in result["history"]], label=name, color=colors[name])
    axes[1].plot(epochs, [row["validation_accuracy"] for row in result["history"]], label=name, color=colors[name])
axes[0].set(title="Validation loss", xlabel="epoch", ylabel="cross-entropy")
axes[1].set(title="Validation accuracy", xlabel="epoch", ylabel="accuracy", ylim=(0.3, 1.02))
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend()
fig.savefig(OUTPUT_DIR / "optimizer_learning_curves.png", dpi=170)
plt.close(fig)

SGD {'best_epoch': 80, 'best_validation_loss': 0.7804059386253357, 'test_loss': 0.7842320203781128, 'test_accuracy': 0.5166666507720947}
Momentum {'best_epoch': 80, 'best_validation_loss': 0.018139410763978958, 'test_loss': 0.017787622287869453, 'test_accuracy': 1.0}
Adam {'best_epoch': 80, 'best_validation_loss': 0.003953812178224325, 'test_loss': 0.003598979441449046, 'test_accuracy': 1.0}


## 5. Save the registered evidence

In [6]:
results = {
    "seed": SEED,
    "protocol": {
        "task": "three-class synthetic spiral",
        "split_sizes": {"train": len(train_y), "validation": len(val_y), "test": len(test_y)},
        "architecture": "2-32-32-3 tanh MLP",
        "epochs": 80,
        "batch_size": 54,
        "selection": "lowest validation cross-entropy",
        "learning_rates": {"SGD": 0.08, "Momentum": 0.05, "Adam": 0.01},
    },
    "quadratic": {"paths": quadratic_paths, "final_loss": quadratic_final_loss},
    "batch_noise": batch_noise,
    "optimizers": optimizer_results,
}
(OUTPUT_DIR / "experiment_results.json").write_text(json.dumps(results, indent=2))
print("Wrote", OUTPUT_DIR / "experiment_results.json")

Wrote outputs/experiment_results.json
